# Experiment 4b – Audio explainability (clean)

Tento notebook je uprataná verzia pre spustenie explainability nad zvukovou modalitou.

Pred spustením uprav v bunke **CONFIG** cesty k súborom:
- `SAMPLES_PATH`
- `MODEL_PATH`

Potom spúšťaj bunky zhora nadol.


In [ ]:
import os
import pickle
import random
from dataclasses import dataclass

import cv2
import librosa
import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.nn as nn
from sklearn.model_selection import train_test_split
from torch.utils.data import Dataset
from torchvision import models, transforms


## Config


In [ ]:
SAMPLES_PATH = "/content/samples.pkl"  # or /content/samples.pkl
MODEL_PATH = "/content/best_model_v2.pth"            # or /content/drive/MyDrive/best_model_v2.pth
RESULTS_DIR = "/content/results/audio_explain_clean"
RANDOM_STATE = 42
THRESHOLD = 0.5
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

os.makedirs(RESULTS_DIR, exist_ok=True)

# MODEL DEFINITION
class VisualBranch(nn.Module):
    def __init__(self, output_dim=256):
        super().__init__()
        efficientnet = models.efficientnet_b0(weights='IMAGENET1K_V1')
        self.features = nn.Sequential(*list(efficientnet.children())[:-1])
        self.fc = nn.Linear(1280, output_dim)
        self.relu = nn.ReLU()

    def forward(self, x):
        x = self.features(x)
        x = x.flatten(1)
        x = self.relu(self.fc(x))
        return x

class AudioBranch(nn.Module):
    def __init__(self, input_dim=40, hidden_dim=128, output_dim=256):
        super().__init__()
        self.lstm = nn.LSTM(
            input_size=input_dim,
            hidden_size=hidden_dim,
            num_layers=2,
            batch_first=True,
            dropout=0.3,
        )
        self.fc = nn.Linear(hidden_dim, output_dim)
        self.relu = nn.ReLU()

    def forward(self, x):
        x = x.permute(0, 2, 1)
        out, _ = self.lstm(x)
        out = out[:, -1, :]
        out = self.relu(self.fc(out))
        return out

class DeepfakeDetector(nn.Module):
    def __init__(self):
        super().__init__()
        self.visual = VisualBranch(output_dim=256)
        self.audio = AudioBranch(output_dim=256)
        self.fusion = nn.Sequential(
            nn.Linear(512, 256),
            nn.ReLU(),
            nn.Dropout(0.5),
            nn.Linear(256, 1),
            nn.Sigmoid(),
        )

    def forward(self, face, mfcc):
        v = self.visual(face)
        a = self.audio(mfcc)
        combined = torch.cat([v, a], dim=1)
        return self.fusion(combined)

# DATASET
augment = transforms.Compose([
    transforms.RandomHorizontalFlip(),
    transforms.ColorJitter(brightness=0.2, contrast=0.2),
])

class DeepfakeDatasetAug(Dataset):
    def __init__(self, samples, augment_enabled=False):
        self.samples = samples
        self.augment_enabled = augment_enabled

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        s = self.samples[idx]
        face = s['face'].float()
        if self.augment_enabled:
            face = augment(face)
        mfcc = torch.tensor(s['mfcc']).float()
        label = torch.tensor(s['label']).float()
        return face, mfcc, label

# LOAD SPLIT + MODEL
with open(SAMPLES_PATH, 'rb') as f:
    all_samples = pickle.load(f)

# Use the same split logic as the 70/15/15 branch from the notebook.
train_s, temp_s = train_test_split(all_samples, test_size=0.3, random_state=RANDOM_STATE)
val_s, test_s = train_test_split(temp_s, test_size=0.5, random_state=RANDOM_STATE)

test_dataset = DeepfakeDatasetAug(test_s, augment_enabled=False)

model = DeepfakeDetector().to(DEVICE)
state = torch.load(MODEL_PATH, map_location=DEVICE)
model.load_state_dict(state)
model.eval()

print(f"Loaded {len(all_samples)} total samples")
print(f"Split sizes -> train: {len(train_s)}, val: {len(val_s)}, test: {len(test_s)}")
print(f"Using device: {DEVICE}")

# AUDIO EXPLAINABILITY
def audio_saliency_for_sample(model, dataset, sample_idx=0, device=DEVICE):
    model.eval()
    face, mfcc, label = dataset[sample_idx]
    face = face.unsqueeze(0).to(device)
    mfcc = mfcc.unsqueeze(0).to(device)
    mfcc.requires_grad_(True)

    output = model(face, mfcc).squeeze()
    model.zero_grad()
    output.backward()

    grads = mfcc.grad.detach().cpu().numpy()[0]
    mfcc_np = mfcc.detach().cpu().numpy()[0]
    saliency = np.abs(grads)
    saliency = saliency / (saliency.max() + 1e-8)

    pred_prob = float(output.detach().cpu().item())
    pred_label = 1 if pred_prob > THRESHOLD else 0
    true_name = "Fake" if int(label.item()) == 1 else "Real"
    pred_name = "Fake" if pred_label == 1 else "Real"
    return mfcc_np, saliency, true_name, pred_name, pred_prob


def plot_audio_saliency(mfcc_np, saliency, true_name, pred_name, pred_prob, sample_idx=None, save_path=None):
    plt.figure(figsize=(12, 8))
    plt.subplot(2, 1, 1)
    plt.imshow(mfcc_np, aspect='auto', origin='lower')
    plt.colorbar()
    plt.title(f"MFCC vstup | sample={sample_idx} | true={true_name} | pred={pred_name} | prob_fake={pred_prob:.4f}")
    plt.ylabel("MFCC koeficient")

    plt.subplot(2, 1, 2)
    plt.imshow(saliency, aspect='auto', origin='lower', cmap='hot')
    plt.colorbar()
    plt.title("Saliency mapa nad MFCC vstupom")
    plt.xlabel("Časový krok")
    plt.ylabel("MFCC koeficient")
    plt.tight_layout()
    if save_path:
        plt.savefig(save_path, dpi=200, bbox_inches='tight')
    plt.show()


def plot_audio_saliency_time_importance(saliency, sample_idx=None, true_name=None, pred_name=None, pred_prob=None, save_path=None):
    time_importance = saliency.mean(axis=0)
    plt.figure(figsize=(10, 4))
    plt.plot(time_importance)
    plt.xlabel("Časový krok")
    plt.ylabel("Priemerná dôležitosť")
    plt.title(f"Časová dôležitosť | sample={sample_idx} | true={true_name} | pred={pred_name} | prob_fake={pred_prob:.4f}")
    plt.grid(True)
    plt.tight_layout()
    if save_path:
        plt.savefig(save_path, dpi=200, bbox_inches='tight')
    plt.show()

# SELECT SAMPLES
correct_real = []
correct_fake = []
wrong_samples = []

with torch.no_grad():
    for idx in range(len(test_dataset)):
        face, mfcc, label = test_dataset[idx]
        face = face.unsqueeze(0).to(DEVICE)
        mfcc = mfcc.unsqueeze(0).to(DEVICE)
        prob_fake = model(face, mfcc).squeeze().item()
        pred = 1 if prob_fake > THRESHOLD else 0
        true = int(label.item())
        info = {"idx": idx, "true": true, "pred": pred, "prob_fake": prob_fake}
        if pred == true:
            if true == 0 and len(correct_real) < 2:
                correct_real.append(info)
            elif true == 1 and len(correct_fake) < 2:
                correct_fake.append(info)
        else:
            if len(wrong_samples) < 2:
                wrong_samples.append(info)
        if len(correct_real) >= 2 and len(correct_fake) >= 2 and len(wrong_samples) >= 2:
            break

print("=== Správne klasifikované REAL ===")
for s in correct_real:
    print(s)
print("\n=== Správne klasifikované FAKE ===")
for s in correct_fake:
    print(s)
print("\n=== Chybné vzorky ===")
for s in wrong_samples:
    print(s)

selected_samples = correct_real + correct_fake + wrong_samples

for s in selected_samples:
    idx = s['idx']
    mfcc_np, saliency, true_name, pred_name, pred_prob = audio_saliency_for_sample(model, test_dataset, idx)
    base = f"idx_{idx}_true_{true_name}_pred_{pred_name}"
    plot_audio_saliency(
        mfcc_np, saliency, true_name, pred_name, pred_prob,
        sample_idx=idx,
        save_path=os.path.join(RESULTS_DIR, f"audio_saliency_{base}.png")
    )
    plot_audio_saliency_time_importance(
        saliency,
        sample_idx=idx,
        true_name=true_name,
        pred_name=pred_name,
        pred_prob=pred_prob,
        save_path=os.path.join(RESULTS_DIR, f"audio_time_{base}.png")
    )

print(f"\nHotovo. Výstupy sú uložené v: {RESULTS_DIR}")
